# Day 57 — Security, data privacy & ethical considerations
Objectives:
- Identify PII and sensitive attributes.
- Basic de-identification and minimization strategies.
- Ethical considerations, fairness checks, and documentation.
Note: This notebook demonstrates lightweight checks; real programs require legal/policy review.

In [ ]:
import re, pandas as pd
from pathlib import Path
sample = pd.DataFrame({
    'name': ['Alice Smith','Bob Jones'],
    'email': ['alice@example.com','bob@company.org'],
    'phone': ['+1-415-555-1212','(212) 555-9898'],
    'notes': ['Met on 2025-01-01','Lives near 5th Ave']
})
sample


## PII detection (simple regex demo)
Caution: regex is imperfect; use specialized tools for robust detection.

In [ ]:
EMAIL_RE = re.compile(r'[\w.%-]+@[\w.-]+\.[A-Za-z]{2,}')
PHONE_RE = re.compile(r'(?:\+?\d{1,3}[-.\s]?)?(?:\(\d{3}\)|\d{3})[-.\s]?\d{3}[-.\s]?\d{4}')
def find_pii(s: str) -> dict:
    emails = EMAIL_RE.findall(s)
    phones = PHONE_RE.findall(s)
    return {'emails': emails, 'phones': phones}

sample['pii'] = sample.apply(lambda r: {
    'email': EMAIL_RE.findall(r['email']),
    'phone': PHONE_RE.findall(r['phone']),
    'notes': find_pii(r['notes'])
}, axis=1)
sample[['pii']]


## De-identification strategies
- Remove direct identifiers (name, email, phone).
- Pseudonymize with stable hashes.
- Mask partial values.
- Limit retention and access (data minimization).
Below: pseudonymize names with a salted hash.

In [ ]:
import hashlib
SALT = b'secret-salt'  # store securely via environment vars/secret manager
def pseudo(value: str) -> str:
    h = hashlib.sha256(SALT + value.encode()).hexdigest()[:10]
    return f'id_{h}'

redacted = sample.copy()
redacted['name_pseudo'] = redacted['name'].map(pseudo)
redacted = redacted.drop(columns=['name','email','phone'])
redacted


## Fairness and bias (brief)
- Check subgroup performance metrics (e.g., by sex, race, age bucket).
- Avoid using protected attributes directly unless justified; document rationale.
- Consider disparate impact, calibration across groups.
- Provide model cards and data statements.

## Learner exercises and progressive hints

1. Build a DataFrame PII scanner covering column names and free text.
2. Add a function that masks email addresses and phone numbers in text.
3. Simulate subgroup precision and recall for a classifier and compare groups.
4. Draft a one-page data-ethics checklist for your project.

### Progressive hints

1. Return finding type, row identifier, column, and a safe count—do not log the
   raw matched value. Test false-positive and missed-format cases.
2. Preserve only the minimum structure needed for debugging and ensure repeated
   substitutions do not reveal the original.
3. Include group support and positive-label counts beside metrics. Avoid a
   conclusion when groups are too small for a stable estimate.
4. Name intended use, excluded use, affected people, owners, data rights,
   retention, access, monitoring, appeals, and incident response.

### Additional mastery practice

Combine data minimization, access control, threat modeling, privacy limits, fairness uncertainty, and incident response. Detection or masking alone is not protection.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

5. **Threat modeling:** Create a data-flow diagram for collection, notebook, artifacts, API, logs, and backups. For each boundary, identify asset, actor, threat, control, residual risk, and owner.
   **Progressive hint:** Include accidental exposure and insider misuse, not only external attackers. Trace data copies and retention through every stage.
6. **Re-identification reasoning:** Generalize a small dataset to satisfy a chosen k-anonymity target, then demonstrate why k-anonymity does not prevent attribute disclosure or attacks using outside information.
   **Progressive hint:** Group quasi-identifiers, inspect equivalence-class sizes and sensitive value diversity, and measure utility loss.
7. **Fairness uncertainty:** Bootstrap subgroup precision and recall, show confidence intervals and support, and compare a gap with a ratio. Explain what to do when one group's denominator is nearly zero.
   **Progressive hint:** Resample at the independent entity level when rows repeat. Undefined metrics should remain undefined rather than being forced to zero.
8. **Incident response:** Simulate discovering raw emails in a committed notebook output. Write the containment, notification, credential review, history cleanup decision, verification, and prevention steps.
   **Progressive hint:** Preserve a restricted incident record, stop further sharing, and assume copied history may exist. Redaction from the latest commit alone is insufficient.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.


In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 5 — Threat modeling


# Practice 6 — Re-identification reasoning


# Practice 7 — Fairness uncertainty


# Practice 8 — Incident response
